In [ ]:
import requests

OLLAMA_HOST = "http://localhost:11434"
OLLAMA_MODEL = "qwen2.5:1.5b"

# Vérifie qu'Ollama est lancé et que le modèle est bien téléchargé.
# (remplace la vérification CUDA/nvidia-smi : plus besoin de GPU, Ollama tourne sur CPU)
try:
    res = requests.get(f"{OLLAMA_HOST}/api/tags", timeout=5)
    res.raise_for_status()
    available = {m["name"] for m in res.json().get("models", [])}
    if OLLAMA_MODEL in available or f"{OLLAMA_MODEL}:latest" in available:
        print(f"Ollama OK — modèle '{OLLAMA_MODEL}' disponible.")
    else:
        print(f"Modèle '{OLLAMA_MODEL}' non trouvé. Lance : ollama pull {OLLAMA_MODEL}")
except requests.exceptions.RequestException as exc:
    print(f"Impossible de contacter Ollama sur {OLLAMA_HOST} ({exc}). Lance : ollama serve")


In [ ]:
!pip install -U langchain-text-splitters
!pip install langchain-huggingface
!pip install langchain_chroma
!pip install torch
!pip install rouge-score
!pip install requests


In [ ]:
from datasets import load_dataset

dataset = load_dataset("uriel/Maathis_Ohada_dataset")


In [ ]:
dataset

In [ ]:
from langchain_core.documents import Document
import pandas as pd

documents = []

for row in dataset["train"]:
    content = f"""
    title : {row['title']}
    content : {row['content']}
    details : {row['details']}
    """

    documents.append(
        Document(
            page_content=content,
            metadata={
                "title": row["title"],
                "content": row["content"],
                "details": row["details"]
            }
        )
    )


In [ ]:
from langchain_huggingface.embeddings import HuggingFaceEmbeddings

embedding = HuggingFaceEmbeddings(model_name="BAAI/bge-small-en-v1.5", encode_kwargs={"normalize_embeddings" : True})


In [ ]:
from langchain_chroma import Chroma

vectorstore = Chroma.from_documents(
    documents=documents,
    embedding=embedding,
    persist_directory="./chroma_db"
)


In [ ]:
def call_ollama(prompt: str, max_new_tokens: int = 100) -> str:
    response = requests.post(
        f"{OLLAMA_HOST}/api/generate",
        json={
            "model": OLLAMA_MODEL,
            "prompt": prompt,
            "stream": False,
            "options": {"num_predict": max_new_tokens, "temperature": 0},
        },
        timeout=300,
    )
    response.raise_for_status()
    return response.json()["response"].strip()


In [ ]:
from langchain_core.prompts import PromptTemplate

prompt_template = PromptTemplate.from_template("""Tu es un assistant chargé de répondre à des questions. Utilises les éléments de contexte récupérés ci-dessous pour répondre à la question. Si tu ne connais pas la réponse, dis simplement que tu ne la connais pas. Limites ta réponse à trois phrases maximum et restes concis.
Question: {question}
Context: {context}
Answer: """)

def rag_pipeline(query):
    retrieved_docs = vectorstore.similarity_search(
        query,
        k=2
    )

    context = "\n\n".join(
        doc.page_content[:3000]
        for doc in retrieved_docs
    )

    prompt = prompt_template.format(
        question=query,
        context=context
    )

    response = call_ollama(prompt)

    return response.strip(), retrieved_docs


In [ ]:
query = """
Les délibérations du tribunal arbitral sont-elles publiques ou secrètes ?
"""

response, retrieved_docs = rag_pipeline(query)
print(response)


In [ ]:
from IPython.display import display, HTML

display(HTML(f"""
<div style="
    border: 1px solid #ddd;
    border-radius: 12px;
    padding: 20px;
    margin: 15px 0;
    background-color: #f8f9fa;
    font-family: Arial, sans-serif;
">
    <h3 style="margin-top: 0;">🤖 Réponse de l'assistant</h3>

    <div style="
        background-color: white;
        border-radius: 8px;
        padding: 15px;
        margin-top: 20px;
        line-height: 1.6;
        border-left: 4px solid ;
    ">
        {response}
    </div>
</div>
"""))


In [ ]:
import re

def extraire_reference_texte(texte):
    correspondance = re.search(r"ARTICLE\s+(\d+)\s+([A-Z]{2,}?)(?=ARTICLE|\s|$)", texte.upper())
    if correspondance:
        return correspondance.group(2), correspondance.group(1)
    return None, None

def extraire_reference(document):
    return extraire_reference_texte(document.metadata["details"])


In [ ]:
import os

if os.path.exists("Test.csv"):
    test = pd.read_csv("Test.csv")
else:
    test = pd.DataFrame([
        {"ID": "Q1", "Question": "Les délibérations du tribunal arbitral sont-elles publiques ou secrètes ?"}
    ])

lignes = []
for _, row in test.iterrows():
    reponse, retrieved_docs = rag_pipeline(row["Question"])
    lignes.append({"ID": row["ID"], "Answer": reponse})

soumission = pd.DataFrame(lignes)
soumission.to_csv("submission.csv", index=False)
soumission


In [ ]:
from rouge_score import rouge_scorer

scorer = rouge_scorer.RougeScorer(["rouge1"], use_stemmer=True)

def calculer_metrique(reponse, document_reference, reponse_reference, document_reference_reference):
    rouge1 = scorer.score(reponse_reference, reponse)["rouge1"].fmeasure
    exactitude_document = float(document_reference == document_reference_reference)
    return 0.5 * rouge1 + 0.5 * exactitude_document


In [ ]:
evaluation_reference = pd.read_csv("Eval.csv")

resultats = []
for _, row in evaluation_reference.iterrows():
    reponse, retrieved_docs = rag_pipeline(row["Question"])
    document_reference, numero_article = extraire_reference(retrieved_docs[0])
    score = calculer_metrique(reponse, document_reference, row["Answer"], row["Act_Title"])
    resultats.append({
        "ID": row["ID"],
        "reponse": reponse,
        "acte_predit": document_reference,
        "acte_attendu": row["Act_Title"],
        "score": score,
    })

evaluation = pd.DataFrame(resultats)
evaluation.to_csv("evaluation.csv", index=False)
evaluation
